# ComfyUI + LTX 2.3 GGUF on Kaggle
Base: pogscafe (2202) + Lightricks/ComfyUI-LTXVideo (3.9k)

**Run All:** Cell 1 xong → Cell 2 tunnel → Cell 3 copy URL → Cell 4 keep alive
---

## Cell 1: Install + Download + Start

In [ ]:
%%time
import os, sys, subprocess, threading, time, json, requests
from datetime import datetime

HOME = '/kaggle/working'
COMFY = f'{HOME}/ComfyUI'
VENV = f'{HOME}/venv'
os.chdir(HOME)

print('=== B1: virtualenv ===')
!pip install -q virtualenv
if not os.path.exists(VENV):
    !virtualenv {VENV} -p $(which python3.10)
print('venv OK')

PIP = f'{VENV}/bin/pip'
PYTHON = f'{VENV}/bin/python3.10'  # Dung tr?c ti?p python3.10, kh?i symlink loop!

print('=== B2: Clone ComfyUI ===')
COMFY_COMMIT = '7fc3ccdcc2fb1f20c4b7dd4aca374db952fd66df'
if not os.path.exists(COMFY):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
os.chdir(COMFY)
!git checkout {COMFY_COMMIT} 2>/dev/null
!{PIP} install -q -r requirements.txt
print('ComfyUI OK')

print('=== B3: Custom nodes ===')
os.chdir(f'{COMFY}/custom_nodes')
for url,name in [
    ('https://github.com/Lightricks/ComfyUI-LTXVideo.git','ComfyUI-LTXVideo'),
    ('https://github.com/logtd/ComfyUI-LTXTricks.git','ComfyUI-LTXTricks'),
    ('https://github.com/city96/ComfyUI-GGUF.git','ComfyUI-GGUF'),
    ('https://github.com/ltdrdata/ComfyUI-Manager.git','ComfyUI-Manager'),
]:
    if not os.path.exists(name):
        !git clone {url}
print('Nodes OK')

print('=== B4: Models symlink -> /tmp ===')
!mkdir -p /tmp/models/{unet,clip,vae}
for d in ['unet','clip','vae']:
    src = f'{COMFY}/models/{d}'
    dst = f'/tmp/models/{d}'
    if os.path.islink(src) or os.path.exists(src):
        !rm -rf {src}
    !ln -sf {dst} {src}

print('=== B5: Download LTX-2.3 GGUF (12.4 GB, ~5 phut) ===')
os.chdir('/tmp/models/unet')
f = 'LTX-2.3-22B-distilled-1.1-Q2_K.gguf'
if not os.path.exists(f):
    !wget -c 'https://huggingface.co/QuantStack/LTX-2.3-GGUF/resolve/main/LTX-2.3-distilled-1.1/LTX-2.3-22B-distilled-1.1-Q2_K.gguf' -O '{f}'
print(f'Model: {os.path.getsize(f)/1e9:.1f} GB')

print('=== B6: Download VAE ===')
os.chdir('/tmp/models/vae')
if not os.path.exists('ltx-vae.safetensors'):
    # Dung VAE tu Comfy-Org (chac chan dung)
    !wget -c 'https://huggingface.co/Comfy-Org/ltx-2/resolve/main/split_files/vae/ltx-vae-1.0.safetensors' -O 'ltx-vae.safetensors'
print('VAE OK')

print('=== B7: Download text encoder (Gemma 3-12B) ===')
print('NOTE: Gemma 3-12B qua lon cho T4 16GB (9-13GB)')
print('Se dung CPU offload hoac Gemma-2B neu ho tro')
print('Skip text encoder - de ComfyUI tu dong tai khi chay')

print('=== B8: Start ComfyUI headless ===')
os.chdir(COMFY)
!pkill -f main.py 2>/dev/null
time.sleep(2)
proc = subprocess.Popen(
    [PYTHON, 'main.py', '--headless', '--port', '8188', '--listen', '127.0.0.1', '--highvram'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

print('Waiting for ComfyUI API...')
for i in range(40):
    time.sleep(3)
    try:
        r = requests.get('http://127.0.0.1:8188/object_info', timeout=2)
        if r.status_code == 200:
            print(f'ComfyUI API ready after {i*3}s')
            break
    except:
        pass
else:
    print('WARNING: ComfyUI API may not be ready')

print()
print('=== CELL 1 DONE! Go to Cell 2 ===')

## Cell 2: Tunnel Pinggy

In [ ]:
# Chay tunnel trong background thread
TUNNEL_URL = None

def run_tunnel():
    global TUNNEL_URL
    p = subprocess.Popen(
        ['ssh', '-p', '443', '-R0:localhost:8188', 'a.pinggy.io'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True
    )
    for line in p.stdout:
        if 'https://' in line:
            i = line.find('https://')
            TUNNEL_URL = line[i:].strip().split()[0]
            open(f'{HOME}/url.txt', 'w').write(TUNNEL_URL)
            print(f'TUNNEL URL: {TUNNEL_URL}')
            break

threading.Thread(target=run_tunnel, daemon=True).start()

print('Dang tao tunnel...')
for i in range(30):
    time.sleep(5)
    if TUNNEL_URL:
        break
    try:
        TUNNEL_URL = open(f'{HOME}/url.txt').read().strip()
    except:
        pass

if TUNNEL_URL:
    print(f'\nCopy URL nay: {TUNNEL_URL}')
else:
    print('Khong co URL, chay lai cell nay')

## Cell 3: Push URL to Gist

An paste URL vao o ben duoi, roi Run cell nay

In [ ]:
# Paste URL tu Cell 2 vao day:
TUNNEL_URL = ""  # <-- PASTE URL HERE

if not TUNNEL_URL:
    print('Chua co URL! Chay Cell 2 truoc.')
else:
    GIST_ID = "8da27f2e6e0d8809a043712cd90f9237"
    data = {}
    try:
        r = requests.get(f'https://api.github.com/gists/{GIST_ID}', timeout=5)
        if r.status_code == 200:
            c = r.json()['files']['kaggle_backends.json']['content']
            data = json.loads(c) if c.strip() else {}
    except:
        pass
    
    data['kaggle-a'] = {
        'url': TUNNEL_URL,
        'status': 'online',
        'updated': datetime.now().isoformat(),
        'capabilities': ['t2v', 'i2v', 'v2v'],
        'gpu': 't4'
    }
    
    r = requests.patch(f'https://api.github.com/gists/{GIST_ID}',
        json={'files': {'kaggle_backends.json': {'content': json.dumps(data, indent=2)}}},
        timeout=10
    )
    
    if r.status_code == 200:
        print('Gist OK! Hermes se detect backend.')
    else:
        print(f'Gist error: {r.status_code}')

## Cell 4: Keep Alive

In [ ]:
try:
    while True:
        time.sleep(60)
        print(f'[alive] {datetime.now().strftime("%H:%M:%S")}')
except:
    print('Stopped')